# Image Model Training (5-Fold Cross Validation)

**ISIC 2024 – Skin Cancer Detection with 3D-TBP**

This notebook trains an image-based classifier using five-fold Stratified Group Cross Validation. The same training pipeline can be used with different backbone architectures, including ResNet18 and EfficientNet-B0.

Each fold produces:

- a trained model,
- training history,
- out-of-fold predictions,
- and validation metrics.

## Training Pipeline

The training procedure follows these steps:

1. Load the ISIC metadata.
2. Create five patient-level cross-validation folds.
3. Build the image datasets and dataloaders.
4. Train one model per fold.
5. Evaluate the validation performance.
6. Save the trained models and Out-of-Fold predictions.

## Configuration

Define the training hyperparameters, dataset paths, and model configuration used throughout the notebook.

In [1]:
# ==========================================================
# Model
# ==========================================================

MODEL_NAME = "resnet18"
# MODEL_NAME = "efficientnet_b0"

# ==========================
# Paths
# ==========================

HDF5_PATH = "/kaggle/input/competitions/isic-2024-challenge/train-image.hdf5"
META_PATH = "/kaggle/input/competitions/isic-2024-challenge/train-metadata.csv"

# ==========================
# Training
# ==========================

N_FOLDS = 5
EPOCHS = 5

BATCH_SIZE = 64

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

# ==========================
# DataLoader
# ==========================

NUM_WORKERS = 2
PIN_MEMORY = True

# ==========================
# Image
# ==========================

IMAGE_SIZE = 224

# ==========================
# Seed
# ==========================

SEED = 42

## Imports

Import the required libraries together with the project modules responsible for dataset loading, model definition, training, evaluation, and prediction.

In [2]:
import torch
import torch.nn as nn
import random
import numpy as np
import pandas as pd
import sys

from torchvision import transforms
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score
from pathlib import Path

In [3]:
OUTPUT_DIR = Path("/kaggle/working")

MODEL_DIR = OUTPUT_DIR / "models"
HISTORY_DIR = OUTPUT_DIR / "history"
OOF_DIR = OUTPUT_DIR / "oof"

MODEL_DIR.mkdir(exist_ok=True)
HISTORY_DIR.mkdir(exist_ok=True)
OOF_DIR.mkdir(exist_ok=True)

In [4]:
sys.path.append(
    "/kaggle/input/datasets/wagneraugustoaff/isic2024-code/src/image"
)

from folds import create_folds
from image_model import ISICModelRsn
from image_model import ISICModelEff
from image_dataset import ISICDataset
from image_utils import seed_everything
from image_train import train_fold
from image_inference import predict

In [5]:
seed_everything(SEED)

In [6]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

## Dataset Preparation

Load the ISIC metadata, create the cross-validation folds, and prepare the image dataset for training.

In [7]:
df = pd.read_csv(
    META_PATH,
    low_memory=False
)

## Cross Validation

Create five patient-level folds using Stratified Group Cross Validation. This strategy preserves the class distribution while ensuring that images from the same patient never appear in both the training and validation sets.

In [ ]:
df = create_folds(df)

## Data Augmentation

Define the training and validation transformations. Data augmentation is applied only during training to improve the model's ability to generalize.

In [8]:
train_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(20),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.1,
        hue=0.02,
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),

])


valid_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),

])

## Fold Training

Train one model for each cross-validation fold and evaluate its performance on the corresponding validation set.

In [ ]:
fold_results = []

for fold in range(N_FOLDS):

    print("=" * 60)
    print(f"Fold {fold}")
    print("=" * 60)

    train_df = df[df["fold"] != fold].reset_index(drop=True)
    valid_df = df[df["fold"] == fold].reset_index(drop=True)

    num_pos = (train_df["target"] == 1).sum()
    num_neg = (train_df["target"] == 0).sum()
    
    # Compute the class weight for the current training split.
    pos_weight = torch.tensor(
        [num_neg / num_pos],
        dtype=torch.float32,
        device=device,
    )

    criterion = torch.nn.BCEWithLogitsLoss(
        pos_weight=pos_weight
    )

    # Apply data augmentation only to the training set.
    train_dataset = ISICDataset(
        dataframe=train_df,
        image_path=HDF5_PATH,
        transform=train_transform,
    )

    valid_dataset = ISICDataset(
        dataframe=valid_df,
        image_path=HDF5_PATH,
        transform=valid_transform,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    valid_loader = DataLoader(
        valid_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    if MODEL_NAME == "resnet18":
        model = ISICModelRsn().to(device)
    
    elif MODEL_NAME == "efficientnet_b0":
        model = ISICModelEff().to(device)
    
    else:
        raise ValueError(f"Unknown model: {MODEL_NAME}")

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    # Train and evaluate the current fold.
    result = train_fold(
        model=model,
        train_loader=train_loader,
        train_df=train_df,
        valid_loader=valid_loader,
        valid_df=valid_df,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        epochs=EPOCHS,
        fold=fold,
        model_path=f"/kaggle/working/best_fold{fold}.pth",
        history_path=f"/kaggle/working/history_fold{fold}.csv",
        oof_path=f"/kaggle/working/oof_fold{fold}.csv"
    )

    fold_results.append(result)

## Cross-Validation Summary

Aggregate the fold-level metrics and compute the final Out-of-Fold performance.

In [ ]:
print("\nTraining finished\n")

for result in fold_results:

    print(
        f"Fold {result['fold']}: "
        f"Best epoch = {result['best_epoch'] + 1}, "
        f"Best pAUC = {result['best_pauc']:.5f}"
    )

mean_pauc = np.mean(
    [r["best_pauc"] for r in fold_results]
)

print(f"\nMean pAUC: {mean_pauc:.5f}")

## Results

Display the validation performance for every fold together with the overall Out-of-Fold metrics.

In [ ]:
results_df = pd.DataFrame(fold_results)

results_df.to_csv(
    "/kaggle/working/fold_results.csv",
    index=False,
)

pd.DataFrame(fold_results)